# 01 — EDA and Preprocessing

This notebook loads the Hugging Face Resume-JD dataset, parses prompt-style records into structured Resume/JD fields, performs text cleaning, and runs exploratory data analysis.

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from datasets import load_dataset

pd.set_option('display.max_colwidth', 160)

## Load raw dataset

In [4]:
dataset = load_dataset('facehuggerapoorv/resume-jd-match')
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 6241
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1759
    })
})
{'text': "For the given job description <<Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for our accelerated growth of 300% in the last 3 years with over 5500+ employees globally, with over 30+ locations in the US and global operations in 32 countries. We believe in providing staffing solutions to address the current talent gap  Right Talent  Right Time  Right Place  Right Price and acting as a Career Coach to our consultants.  \nRole: Basel Business AnalystLocation: Washington, D.C.Work Mode: HybridHire Type: 6+ Month Contract (extendable)\nJD: Role Specific Experience: 6+ years of relevant technical and business work experience The Candidates who have worked on Basel related projects in Credit risks or at least are aware of credit r

## Parse prompt records into tabular Resume/JD pairs

The dataset stores examples as prompt text in this structure: `job description <<...>> ... resume: <<...>> ... result`.

In [7]:
PAIR_PATTERN = re.compile(
    r"job description\s*<<(.*?)>>\s*(?:and\s*)?(?:the\s+)?resume\s*:?\s*<<(.*?)>>\s*\.?\s*(?:the\s+result\s+is|$)",
    flags=re.IGNORECASE | re.DOTALL,
)
WHITESPACE_PATTERN = re.compile(r"\s+")
NON_TEXT_PATTERN = re.compile(r"[^a-zA-Z0-9\s\.\,\-\+\#\/]")

def clean_text(text: str) -> str:
    text = text.strip().lower()
    text = NON_TEXT_PATTERN.sub(" ", text)
    text = WHITESPACE_PATTERN.sub(" ", text)
    return text.strip()


def coerce_binary_label(label: str) -> int:
    normalized = str(label).strip().lower()
    if normalized in {"no fit", "0", "false"}:
        return 0
    if normalized in {"potential fit", "good fit", "fit", "1", "true"}:
        return 1
    return int("fit" in normalized and "no fit" not in normalized)

def parse_record(text: str) -> tuple[str, str]:
    match = PAIR_PATTERN.search(str(text))
    if not match:
        raise ValueError("Unable to parse prompt into JD and Resume fields.")
    jd_text = match.group(1)
    resume_text = match.group(2)
    return clean_text(resume_text), clean_text(jd_text)

In [8]:
rows = []
for split_name in dataset.keys():
    split_df = dataset[split_name].to_pandas()
    for _, row in split_df.iterrows():
        resume_text, jd_text = parse_record(row['text'])
        rows.append({
            'split': split_name,
            'resume_text': resume_text,
            'jd_text': jd_text,
            'label': coerce_binary_label(row['label']),
            'raw_label': row['label'],
        })

df = pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)
df['resume_len'] = df['resume_text'].str.len()
df['jd_len'] = df['jd_text'].str.len()
df.head(3)

,split,resume_text,jd_text,label,raw_label,resume_len,jd_len
0,train,summaryhighly motivated sales associate with extensive customer service and sales experience. outgoing sales professional with track record of driving incre...,net2source inc. is an award-winning total workforce solutions company recognized by staffing industry analysts for our accelerated growth of 300 in the last...,0,No Fit,6686,3906
1,train,professional summarycurrently working with caterpillar as an contract employee active in several npi and cpi projects provide technical for cat facilities w...,at salas obrien we tell our clients that were engineered for impact. this passion for making a difference applies just as much to our team as it does to our...,0,No Fit,6526,3804
2,train,"summaryi started my construction career in june of 2017 in jacksonville, florida as an electrical apprentice working full time and attending the jacksonvill...",schweitzer engineering laboratories sel infrastructure defense division seeks a talented individual for a cybersecurity focused lead software engineer posit...,0,No Fit,7205,3174


## Data quality checks

In [9]:
print(f"Shape: {df.shape}")
print(f"Null counts:\n{df[['resume_text', 'jd_text', 'label']].isna().sum()}")
print("Binary label distribution:")
print(df['label'].value_counts(normalize=True).rename('ratio'))

Shape: (7999, 7)
Null counts:
resume_text    0
jd_text        0
label          0
dtype: int64
Binary label distribution:
label
1    0.500063
0    0.499937
Name: ratio, dtype: float64


## EDA — class distribution

In [10]:
class_counts = (
    df.groupby('label', as_index=False)
      .size()
      .assign(label_name=lambda x: x['label'].map({0: 'No Fit', 1: 'Fit'}))
)
fig = px.bar(
    class_counts,
    x='label_name',
    y='size',
    text='size',
    title='Class Distribution (Binary Labels)',
    color='label_name',
    color_discrete_map={'No Fit': '#ef4444', 'Fit': '#22c55e'},
)
fig.update_layout(showlegend=False, template='plotly_white')
fig.show()

## EDA — text length histograms

In [11]:
fig_resume = px.histogram(
    df,
    x='resume_len',
    nbins=40,
    title='Resume Text Length Distribution',
    color_discrete_sequence=['#3b82f6'],
)
fig_resume.update_layout(template='plotly_white')
fig_resume.show()

fig_jd = px.histogram(
    df,
    x='jd_len',
    nbins=40,
    title='Job Description Length Distribution',
    color_discrete_sequence=['#8b5cf6'],
)
fig_jd.update_layout(template='plotly_white')
fig_jd.show()

## Persist cleaned dataset

In [ ]:
out_dir = Path('data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

output_df = df[['resume_text', 'jd_text', 'label']].copy()
output_df.to_csv(out_dir / 'resume_jd_clean.csv', index=False)
output_df.to_parquet(out_dir / 'resume_jd_clean.parquet', index=False)
print(f"Saved cleaned data to: {out_dir.resolve()}")